# *Task 1 — Eksplorasi Awal DataFrame*

In [1]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/refs/heads/master/titanic.csv'
df = pd.read_csv(url)

print("Shape:", df.shape)
print()
print("Tipe data tiap kolom:")
print(df.dtypes)
print()
print("Jumlah null per kolom (diurutkan dari terbanyak):")
print(df.isnull().sum().sort_values(ascending=False))

Shape: (891, 12)

Tipe data tiap kolom:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Jumlah null per kolom (diurutkan dari terbanyak):
Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64


In [2]:
# Statistik deskriptif kolom numerik
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


*Identifikasi nilai tidak wajar:*

- Kolom `Age`: nilai minimum 0.42 kemungkinan valid (bayi), tapi perlu dicatat.
- Kolom `Fare`: nilai minimum 0.0 terlihat tidak wajar, harusnya semua penumpang membayar tiket.
- Kolom `SibSp` dan `Parch`: nilai maksimum cukup tinggi (8 dan 6), berarti ada penumpang dengan keluarga sangat besar.

# *Task 2 — Querying dengan Kondisi Majemuk*

In [3]:
# Filter penumpang kelas 1/2, usia > 30, tidak selamat
kondisi = (
    df['Pclass'].isin([1, 2]) &
    (df['Age'] > 30) &
    (df['Survived'] == 0)
)
hasil = df[kondisi]
print("Penumpang yang memenuhi kriteria:", len(hasil))

Penumpang yang memenuhi kriteria: 94


In [4]:
# Filter penumpang perempuan yang selamat dan usia < 18
perempuan_muda_selamat = df[
    (df['Sex'] == 'female') &
    (df['Survived'] == 1) &
    (df['Age'] < 18)
]
perempuan_muda_selamat[['Name', 'Age', 'Pclass', 'Embarked']]

,Name,Age,Pclass,Embarked
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.00,2,C
10,"Sandstrom, Miss. Marguerite Rut",4.00,3,S
22,"McGowan, Miss. Anna ""Annie""",15.00,3,Q
39,"Nicola-Yarred, Miss. Jamila",14.00,3,C
43,"Laroche, Miss. Simonne Marie Anne Andree",3.00,2,C
58,"West, Miss. Constance Mirium",5.00,2,S
68,"Andersson, Miss. Erna Alexandra",17.00,3,S
84,"Ilett, Miss. Bertha",17.00,2,S
156,"Gilnagh, Miss. Katherine ""Katie""",16.00,3,Q
172,"Johnson, Miss. Eleanor Ileen",1.00,3,S


In [5]:
def kategorikan_usia(age):
    if pd.isna(age):
        return np.nan
    elif age < 12:
        return 'Anak'
    elif age < 18:
        return 'Remaja'
    elif age < 60:
        return 'Dewasa'
    else:
        return 'Lansia'

df['kelompok_usia'] = df['Age'].apply(kategorikan_usia)
print(df['kelompok_usia'].value_counts())

kelompok_usia
Dewasa    575
Anak       68
Remaja     45
Lansia     26
Name: count, dtype: int64


# *Task 3 — Groupby dan Agregasi*

In [6]:
# Survival rate per kelas kabin
survival = df.groupby('Pclass')['Survived'].agg(
    jumlah_penumpang='count',
    jumlah_selamat='sum'
).reset_index()

survival.columns = ['pclass', 'jumlah_penumpang', 'jumlah_selamat']
survival['survival_rate'] = (survival['jumlah_selamat'] / survival['jumlah_penumpang'] * 100).round(2)
print(survival)

   pclass  jumlah_penumpang  jumlah_selamat  survival_rate
0       1               216             136          62.96
1       2               184              87          47.28
2       3               491             119          24.24


In [7]:
# Rata-rata usia dan fare per Pclass dan Sex
rata_rata = df.groupby(['Pclass', 'Sex']).agg(
    rata_usia=('Age', 'mean'),
    rata_fare=('Fare', 'mean')
).round(2)

print(rata_rata)

               rata_usia  rata_fare
Pclass Sex                         
1      female      34.61     106.13
       male        41.28      67.23
2      female      28.72      21.97
       male        30.74      19.74
3      female      21.75      16.12
       male        26.51      12.66


In [8]:
# Jumlah selamat/tidak per kelompok usia pakai crosstab
tabel = pd.crosstab(df['kelompok_usia'], df['Survived'])
tabel.columns = ['Tidak Selamat', 'Selamat']
print(tabel)

               Tidak Selamat  Selamat
kelompok_usia                        
Anak                      29       39
Dewasa                   353      222
Lansia                    19        7
Remaja                    23       22


# *Task 4 — Interpretasi*

Berdasarkan hasil analisis di Task 3, berikut beberapa insight yang ditemukan:

1. *Penumpang kelas 1 memiliki survival rate tertinggi sebesar 62.96%, jauh di atas kelas 3 yang hanya 24.24%.* Ini menunjukkan adanya ketimpangan akses ke sekoci antara kelas kabin.

2. *Rata-rata fare penumpang kelas 1 jauh lebih tinggi dibanding kelas 2 dan 3*, misalnya rata-rata fare laki-laki kelas 1 mencapai sekitar 67.23, sementara kelas 3 hanya sekitar 12.66. Penumpang yang membayar lebih mahal tampaknya mendapat prioritas evakuasi lebih baik.

3. *Kelompok Anak memiliki proporsi selamat yang lebih baik dibanding Dewasa*, dari crosstab terlihat mayoritas anak berhasil selamat, konsisten dengan kebijakan "women and children first" yang diterapkan saat evakuasi Titanic.